In [ ]:
!pip install lightgbm scikit-fuzzy -q

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Path to the dataset folder
BASE_DIR = "SSAQS dataset"  # Update this path to your local dataset folder

# Reads all 7 sensor files for one student and combines into a daily table
def process_student(student_id, base_dir):
    path = os.path.join(base_dir, str(student_id))

    # Daily stress and anxiety ratings - target variable
    try:
        q = pd.read_csv(os.path.join(path, "daily_questions.csv"))
        q['date'] = pd.to_datetime(q['timeStampScheduled'], unit='s').dt.date
        q = q[['date','stress','anxiety']].drop_duplicates('date')
    except:
        return None
    if q.empty:
        return None

    # Heart rate variability - key stress indicator
    try:
        hrv = pd.read_csv(os.path.join(path, "hrv.csv"))
        hrv['date'] = pd.to_datetime(hrv['timestamp']).dt.date
        hrv_daily = hrv.groupby('date').agg(
            hrv_rmssd_mean=('rmssd','mean'),
            hrv_rmssd_std =('rmssd','std'),
            hrv_lf_mean   =('low_frequency','mean'),
            hrv_hf_mean   =('high_frequency','mean'),
        ).reset_index()
    except:
        hrv_daily = pd.DataFrame(columns=['date','hrv_rmssd_mean','hrv_rmssd_std','hrv_lf_mean','hrv_hf_mean'])

    # Blood oxygen level - values below 70 are sensor errors, removed
    try:
        oxy = pd.read_csv(os.path.join(path, "oxygen.csv"))
        oxy['date'] = pd.to_datetime(oxy['timestamp']).dt.date
        oxy = oxy[oxy['value'] > 70]
        oxy_daily = oxy.groupby('date').agg(
            spo2_mean=('value','mean'),
            spo2_min =('value','min'),
        ).reset_index()
    except:
        oxy_daily = pd.DataFrame(columns=['date','spo2_mean','spo2_min'])

    # Total steps taken each day
    try:
        steps = pd.read_csv(os.path.join(path, "steps.csv"))
        steps['date'] = pd.to_datetime(steps['timestamp']).dt.date
        steps_daily = steps.groupby('date').agg(steps_total=('steps','sum')).reset_index()
    except:
        steps_daily = pd.DataFrame(columns=['date','steps_total'])

    # Physical activity - ratio of active minutes to total minutes per day
    try:
        act = pd.read_csv(os.path.join(path, "activity_level.csv"))
        act['date'] = pd.to_datetime(act['timestamp']).dt.date
        act['is_active'] = act['level'].isin(['LIGHTLY_ACTIVE','FAIRLY_ACTIVE','VERY_ACTIVE'])
        act_daily = act.groupby('date').agg(
            active_minutes=('is_active','sum'),
            total_minutes =('is_active','count'),
        ).reset_index()
        act_daily['active_ratio'] = act_daily['active_minutes'] / act_daily['total_minutes']
    except:
        act_daily = pd.DataFrame(columns=['date','active_minutes','total_minutes','active_ratio'])

    # Sleep quality score and deep sleep duration
    try:
        sleep = pd.read_csv(os.path.join(path, "sleep.csv"))
        sleep['date'] = pd.to_datetime(sleep['timestamp']).dt.date
        sleep_daily = sleep[['date','overall_score','deep_sleep_in_minutes']].copy()
        sleep_daily.columns = ['date','sleep_score','deep_sleep_min']
        sleep_daily = sleep_daily.drop_duplicates('date')
    except:
        sleep_daily = pd.DataFrame(columns=['date','sleep_score','deep_sleep_min'])

    # Fitbit built-in stress score - failed calculations removed
    try:
        st = pd.read_csv(os.path.join(path, "stress.csv"))
        st['date'] = pd.to_datetime(st['DATE']).dt.date
        st = st[st['CALCULATION_FAILED'] == False]
        stress_daily = st[['date','STRESS_SCORE']].rename(columns={'STRESS_SCORE':'fitbit_stress'})
    except:
        stress_daily = pd.DataFrame(columns=['date','fitbit_stress'])

    # Merge all sensor data by date
    merged = q.copy()
    for part in [hrv_daily, oxy_daily, steps_daily, act_daily, sleep_daily, stress_daily]:
        if not part.empty:
            merged = merged.merge(part, on='date', how='left')

    merged['student_id'] = int(student_id)
    return merged


# Loop through all student folders and combine into one dataframe
student_ids = sorted([d for d in os.listdir(BASE_DIR)
                      if os.path.isdir(os.path.join(BASE_DIR, d)) and d.isdigit()], key=int)
print(f"Found {len(student_ids)} students")

all_dfs = []
for sid in student_ids:
    result = process_student(sid, BASE_DIR)
    if result is not None:
        all_dfs.append(result)
        print(f"Student {sid}: {len(result)} days")

df = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal rows: {len(df)}")

In [ ]:
# Sensor columns used as input features
SENSOR_COLS = ['hrv_rmssd_mean','hrv_rmssd_std','hrv_lf_mean','hrv_hf_mean',
               'spo2_mean','spo2_min','steps_total','active_ratio',
               'active_minutes','sleep_score','deep_sleep_min','fitbit_stress']

# Fill missing values with each student's own average
# If still missing, fill with the overall median
for col in SENSOR_COLS:
    if col in df.columns:
        df[col] = df.groupby('student_id')[col].transform(lambda x: x.fillna(x.mean()))
        df[col] = df[col].fillna(df[col].median())

# Create stress labels per student (0=Low, 1=Medium, 2=High)
# Labels are based on each student's own stress distribution, not a global threshold
def label_stress(s):
    low  = s.quantile(0.33)
    high = s.quantile(0.67)
    return s.apply(lambda x: 0 if x <= low else (2 if x >= high else 1))

df['stress_label'] = df.groupby('student_id')['stress'].transform(label_stress)

# Remove rows with missing stress or HRV values
df = df.dropna(subset=['stress','hrv_rmssd_mean'])

print(f"Shape: {df.shape}")
print(f"Students: {df['student_id'].nunique()}")
print(df['stress_label'].value_counts())

In [ ]:
# Input features from Fitbit wearable device: HRV, SpO2, steps, activity, sleep, stress score
FEATURES = ['hrv_rmssd_mean','hrv_rmssd_std','hrv_lf_mean','hrv_hf_mean',
            'spo2_mean','spo2_min','steps_total','active_ratio',
            'active_minutes','sleep_score','deep_sleep_min','fitbit_stress']

# X = input features (sensor data), scaled between 0 and 1
# y = target labels (0=Low stress, 1=Medium stress, 2=High stress)
# groups = student IDs, used for Leave-One-Student-Out validation
X      = MinMaxScaler().fit_transform(df[FEATURES].values)
y      = df['stress_label'].values
groups = df['student_id'].values

print("Ready! X shape:", X.shape)

In [ ]:
# Random Forest: builds many decision trees and combines their predictions
# LOSO (Leave-One-Student-Out): train on 34 students, test on 1, repeat 35 times
# Ensures the model is tested on students it has never seen before

print("Training Random Forest with LOSO validation...")

rf_preds, rf_true = [], []

for train_idx, test_idx in LeaveOneGroupOut().split(X, y, groups):
    # Train on all students except one
    clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    clf.fit(X[train_idx], y[train_idx])

    # Test on the left-out student
    rf_preds.extend(clf.predict(X[test_idx]))
    rf_true.extend(y[test_idx])

rf_acc = round(accuracy_score(rf_true, rf_preds), 4)
rf_f1  = round(f1_score(rf_true, rf_preds, average='weighted'), 4)
rf_mae = round(mean_absolute_error(rf_true, rf_preds), 4)

print("=== Random Forest Results ===")
print(f"Accuracy : {rf_acc}  (how often the model is correct)")
print(f"F1-Score : {rf_f1}   (balance between precision and recall)")
print(f"MAE      : {rf_mae}  (lower = predictions closer to true label)")

In [ ]:
# LightGBM: a fast gradient boosting model, effective for tabular sensor data
# Builds trees sequentially - each tree corrects the errors of the previous one

print("Training LightGBM with LOSO validation...")

lgb_preds, lgb_true = [], []

for train_idx, test_idx in LeaveOneGroupOut().split(X, y, groups):
    clf = lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
    clf.fit(X[train_idx], y[train_idx])
    lgb_preds.extend(clf.predict(X[test_idx]))
    lgb_true.extend(y[test_idx])

lgb_acc = round(accuracy_score(lgb_true, lgb_preds), 4)
lgb_f1  = round(f1_score(lgb_true, lgb_preds, average='weighted'), 4)
lgb_mae = round(mean_absolute_error(lgb_true, lgb_preds), 4)

print("=== LightGBM Results ===")
print(f"Accuracy : {lgb_acc}")
print(f"F1-Score : {lgb_f1}")
print(f"MAE      : {lgb_mae}")

In [ ]:
# CNN (Convolutional Neural Network): detects local patterns in sensor data
# Conv1d scans across the 12 features to find stress-related signal combinations
# e.g. low HRV combined with low SpO2 may indicate a stress pattern

class SensorCNN(nn.Module):
    def __init__(self, n_features=12, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            # First convolutional layer: finds basic patterns
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            # Second convolutional layer: finds complex patterns
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            # Reduce to single value per filter
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            # Dropout: randomly turns off 30% of neurons to prevent overfitting
            nn.Dropout(0.3),
            # Final layer: outputs probability for each stress class
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        return self.net(x.unsqueeze(1))

print("CNN model defined successfully!")

In [ ]:
# Train CNN with LOSO validation
# For each student: train on the remaining students, test on that student
# epochs=15: the model sees the training data 15 times

print("Training CNN with LOSO validation...")

cnn_preds, cnn_true = [], []

for i, (train_idx, test_idx) in enumerate(LeaveOneGroupOut().split(X, y, groups)):
    model  = SensorCNN()
    opt    = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit   = nn.CrossEntropyLoss()

    # Convert data to PyTorch tensors
    Xtr    = torch.tensor(X[train_idx], dtype=torch.float32)
    ytr    = torch.tensor(y[train_idx], dtype=torch.long)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

    # Training loop
    for epoch in range(15):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()

    # Evaluate on the left-out student
    model.eval()
    with torch.no_grad():
        Xte = torch.tensor(X[test_idx], dtype=torch.float32)
        cnn_preds.extend(model(Xte).argmax(1).numpy())
        cnn_true.extend(y[test_idx])

    if (i+1) % 5 == 0:
        print(f"  Processed {i+1}/35 students...")

cnn_acc = round(accuracy_score(cnn_true, cnn_preds), 4)
cnn_f1  = round(f1_score(cnn_true, cnn_preds, average='weighted'), 4)
cnn_mae = round(mean_absolute_error(cnn_true, cnn_preds), 4)

print("\n=== CNN Results ===")
print(f"Accuracy : {cnn_acc}")
print(f"F1-Score : {cnn_f1}")
print(f"MAE      : {cnn_mae}")

In [ ]:
# LSTM (Long Short-Term Memory): a recurrent neural network
# Designed for sequential data - remembers patterns over time
# e.g. poor sleep followed by low HRV the next day may indicate high stress

class StressLSTM(nn.Module):
    def __init__(self, input_size=12, hidden_size=64, n_classes=3):
        super().__init__()
        # LSTM layer: processes the 12 features as a sequence
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True)
        # Final classification layer
        self.fc   = nn.Linear(hidden_size, n_classes)
    def forward(self, x):
        out, _ = self.lstm(x.unsqueeze(1))
        return self.fc(out[:, -1, :])  # take output from last timestep

print("Training LSTM with LOSO validation...")

lstm_preds, lstm_true = [], []

for i, (train_idx, test_idx) in enumerate(LeaveOneGroupOut().split(X, y, groups)):
    model  = StressLSTM()
    opt    = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit   = nn.CrossEntropyLoss()
    Xtr    = torch.tensor(X[train_idx], dtype=torch.float32)
    ytr    = torch.tensor(y[train_idx], dtype=torch.long)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

    for epoch in range(15):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        Xte = torch.tensor(X[test_idx], dtype=torch.float32)
        lstm_preds.extend(model(Xte).argmax(1).numpy())
        lstm_true.extend(y[test_idx])

    if (i+1) % 5 == 0:
        print(f"  Processed {i+1}/35 students...")

lstm_acc = round(accuracy_score(lstm_true, lstm_preds), 4)
lstm_f1  = round(f1_score(lstm_true, lstm_preds, average='weighted'), 4)
lstm_mae = round(mean_absolute_error(lstm_true, lstm_preds), 4)

print("\n=== LSTM Results ===")
print(f"Accuracy : {lstm_acc}")
print(f"F1-Score : {lstm_f1}")
print(f"MAE      : {lstm_mae}")

In [ ]:
# Fuzzy Logic: a rule-based system that mimics human reasoning
# Instead of hard thresholds, it uses weighted combination of key signals
# Key advantage: interpretable - the prediction can be explained by the rules

print("Running Fuzzy Logic inference...")

# 3 most clinically meaningful features for stress detection
hrv_idx   = FEATURES.index('hrv_rmssd_mean')  # heart rate variability
spo2_idx  = FEATURES.index('spo2_mean')        # blood oxygen level
sleep_idx = FEATURES.index('sleep_score')      # sleep quality

def fuzzy_predict(hrv, spo2, sleep):
    # Higher HRV = lower stress (relaxed autonomic nervous system)
    # Lower SpO2 = higher stress (poor oxygenation)
    # Lower sleep score = higher stress (poor recovery)
    score = (1 - hrv) * 0.4 + (1 - spo2) * 0.3 + (1 - sleep) * 0.3
    if score < 0.35:   return 0  # Low stress
    elif score < 0.65: return 1  # Medium stress
    else:              return 2  # High stress

fuzzy_preds = np.array([
    fuzzy_predict(row[hrv_idx], row[spo2_idx], row[sleep_idx])
    for row in X
])

fz_acc = round(accuracy_score(y, fuzzy_preds), 4)
fz_f1  = round(f1_score(y, fuzzy_preds, average='weighted'), 4)
fz_mae = round(mean_absolute_error(y, fuzzy_preds), 4)

print("=== Fuzzy Logic Results ===")
print(f"Accuracy : {fz_acc}")
print(f"F1-Score : {fz_f1}")
print(f"MAE      : {fz_mae}")

In [ ]:
# Summarize 

results = {
    'Model':    ['Random Forest', 'LightGBM', 'CNN', 'LSTM', 'Fuzzy Logic'],
    'Accuracy': [rf_acc,  lgb_acc,  cnn_acc,  lstm_acc,  fz_acc],
    'F1-Score': [rf_f1,   lgb_f1,   cnn_f1,   lstm_f1,   fz_f1],
    'MAE':      [rf_mae,  lgb_mae,  cnn_mae,  lstm_mae,  fz_mae],
}
results_df = pd.DataFrame(results)
print("="*55)
print("FINAL MODEL COMPARISON")
print("="*55)
print(results_df.to_string(index=False))

# ── Bar Chart ──
x     = np.arange(len(results['Model']))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))
b1 = ax.bar(x - width, results['Accuracy'], width, label='Accuracy',  color='steelblue', alpha=0.85)
b2 = ax.bar(x,         results['F1-Score'], width, label='F1-Score',  color='seagreen',  alpha=0.85)
b3 = ax.bar(x + width, results['MAE'],      width, label='MAE',       color='tomato',    alpha=0.85)

for bar in [*b1, *b2, *b3]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(results['Model'], fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title('Model Comparison - Stress Prediction from Wearable Sensor Data',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=11)
ax.set_xlabel('Models', fontsize=11)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

# ── Confusion Matrix for best model ──
cm = confusion_matrix(lstm_true, lstm_preds)
fig2, ax2 = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low','Medium','High'],
            yticklabels=['Low','Medium','High'], ax=ax2)
ax2.set_title('LSTM Confusion Matrix', fontsize=13, fontweight='bold')
ax2.set_xlabel('Predicted Label')
ax2.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('lstm_confusion_matrix.png', dpi=150)
plt.show()

# ── Feature Importance (Random Forest) ──
rf_final = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_final.fit(X, y)
importances = pd.Series(rf_final.feature_importances_, index=FEATURES).sort_values()

fig3, ax3 = plt.subplots(figsize=(8, 6))
importances.plot(kind='barh', ax=ax3, color='steelblue', alpha=0.85)
ax3.set_title('Feature Importance - Which sensors matter most?',
              fontsize=13, fontweight='bold')
ax3.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print("\nAll graphs saved: model_comparison.png, lstm_confusion_matrix.png, feature_importance.png")